# Biohub leakage-safe CV — fold 0 pilot

Train on embryo `44b6`, evaluate on an evenly spaced 8-clip pilot subset of embryo `6bba`. This is a short end-to-end runtime and correctness check before full held-out evaluation.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

COMP = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAIN = COMP / 'train'
SUPPORT_CANDIDATES = [Path('/kaggle/input/biohub-tracking-support-pack-50ep-v1'), Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')]
SUPPORT = next((path for path in SUPPORT_CANDIDATES if path.exists()), SUPPORT_CANDIDATES[0])
WORK = Path('/kaggle/working')
REPO = WORK / 'tracking_repo'
assert TRAIN.exists(), TRAIN
assert SUPPORT.exists(), SUPPORT
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SUPPORT / 'repo', REPO)

# The support pack contains wheels, so the notebook remains Internet-off.
packages = ['tracksdata', 'zarr', 'pyscipopt', 'geff', 'geff-spec', 'ilpy', 'polars-runtime-32', 'polars', 'blosc2', 'dask', 'imagecodecs', 'pyarrow', 'rustworkx', 'sqlalchemy', 'donfig', 'numcodecs']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--find-links', str(SUPPORT / 'wheels'), '--force-reinstall', *packages], check=True)
import polars as pl
assert hasattr(pl, 'Float16'), f'Unsupported Polars: {pl.__version__}'
print('Prepared', REPO)


In [ ]:
import json

def embryo_id(name: str) -> str:
    return name.removesuffix('.zarr').split('_', 1)[0]

samples = sorted(path.name.removesuffix('.zarr') for path in TRAIN.glob('*.zarr'))
embryos = sorted({embryo_id(name) for name in samples})
assert embryos == ['44b6', '6bba'], embryos
held_out = '6bba'
held_out_samples = [name for name in samples if embryo_id(name) == held_out]
pilot_size = 8
pilot_indices = [round(i * (len(held_out_samples) - 1) / (pilot_size - 1)) for i in range(pilot_size)]
pilot_test = [held_out_samples[i] for i in pilot_indices]
assert len(pilot_test) == pilot_size and len(set(pilot_test)) == pilot_size
split = [{
    'held_out_embryo': held_out,
    'train': [name for name in samples if embryo_id(name) != held_out],
    'test': pilot_test,
}]
splits_path = REPO / 'dataset_splits.json'
splits_path.write_text(json.dumps(split, indent=2))
print({'train_clips': len(split[0]['train']), 'pilot_test_clips': len(split[0]['test']), 'held_out': held_out})
print('pilot_test:', pilot_test)


In [ ]:
# Train only on 44b6. Eight 6bba clips are used for cheap pilot checkpoint selection.
train_cmd = [
    sys.executable, 'scripts/train_unet_transformer.py',
    '--data-dir', str(TRAIN), '--splits', 'dataset_splits.json', '--split', '0',
    '--epochs', '5', '--max-iters', '50', '--batch-size', '4', '--num-workers', '4', '--single-gpu',
]
subprocess.run(train_cmd, cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src', 'PYTORCH_ALLOC_CONF': 'expandable_segments:True'}, check=True)


In [ ]:
# Predict the same held-out pilot subset and compute the official metric end to end.
predict_cmd = [
    sys.executable, 'scripts/predict_unet_transformer.py',
    '--data-dir', str(TRAIN), '--splits', 'dataset_splits.json', '--split', '0',
    '--weights', 'weights/unet_transformer/split_0/edge_predictor_best.pth',
    '--unet-batch-size', '4', '--det-threshold', '0.99', '--use-ilp', '--evaluate',
]
subprocess.run(predict_cmd, cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src', 'BIOHUB_DATA_DIR': str(TRAIN)}, check=True)
subprocess.run([sys.executable, 'scripts/evaluate.py', '--method', 'unet_transformer', '--split', '0'], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src', 'BIOHUB_DATA_DIR': str(TRAIN)}, check=True)


In [ ]:
# Persist the checkpoint and exact pilot split for the subsequent full-fold run.
summary = {
    'fold': 0, 'pilot': True, 'epochs': 5, 'max_iters': 50,
    'train_embryo': '44b6', 'held_out_embryo': '6bba',
    'weights': 'tracking_repo/weights/unet_transformer/split_0/edge_predictor_best.pth',
    'splits': split,
}
(WORK / 'fold0_pilot_manifest.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
